#Training Pipeline

In [2]:
import copy
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import numpy as np


In [10]:
# 1. SetUp & Device 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
print(f"using device: {device}")

using device: cpu


In [11]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [12]:
df.drop(columns = ['id', 'Unnamed: 32'], inplace=True)

In [13]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [19]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2) #Scikit-learn code
#This divides your dataset into two parts

##Scaling

In [20]:
#The purpose is to bring your input features (columns) onto a similar scale so that a machine learning algorithm can learn more effectively.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
#Why don't we use fit_transform() on X_test?
#This is a very important ML concept called Data Leakage.

#Imagine:
#- Training data has a mean CGPA of 7.5.
#- Testing data has a mean CGPA of 8.5.
#If you independently fit the scaler on testing data, you're using information from your test dataset to preprocess it differently.

#This creates an inconsistent evaluation process.

In [21]:

X_train

array([[-1.21225534, -0.82319299, -1.18824853, ..., -0.7416912 ,
        -0.64514938,  0.49404775],
       [ 0.02456703,  0.31431014,  0.02281569, ...,  0.3062302 ,
        -0.59277454, -0.31306633],
       [ 1.55603328,  1.36268932,  1.4760104 , ...,  0.5001766 ,
        -0.98722254, -1.36870704],
       ...,
       [ 1.06164529, -1.39780797,  0.93657003, ..., -0.46482127,
        -1.82194653, -1.50013938],
       [ 0.11833027,  0.03521144,  0.09858441, ..., -0.273471  ,
        -0.6958875 , -0.54904714],
       [-0.38458165, -0.08205693, -0.41120733, ..., -0.46924996,
        -0.24906466, -0.68346658]])

In [22]:
y_train

524    B
476    B
451    M
371    B
344    B
      ..
345    B
355    B
491    B
448    B
362    B
Name: diagnosis, Length: 455, dtype: object

##Label Encoding


In [23]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)


In [19]:
y_train

array([0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0,
       1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1,
       0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1,
       1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1,
       0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0,
       0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1,
       1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1,
       0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,

##Numpy array to Tensors 


In [24]:
X_train_tensors = torch.from_numpy(X_train)
X_test_tensors = torch.from_numpy(X_test)
y_train_tensors = torch.from_numpy(y_train)
y_test_tensors = torch.from_numpy(y_test)

In [25]:
X_train_tensors.shape

torch.Size([455, 30])

##Defining the model

In [31]:
#You're creating a custom neural network class called mySimplenn using PyTorch.
#This class initializes the weights and bias of a simple neural network (or single linear neuron).
class mySimplenn():    #You're defining your own Python class. Think of it as a blueprint for creating a simple neural network.
    
    def __init__(self, X): #X represents your input training data.
        
        self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True) 
        #Creates a tensor containing random numbers between 0 and 1.
        self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)
        
    def forward(self, X):
        z = torch.matmul(X, self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred
    
    def loss_function(self, y_pred, y):
        #clamp prediction to avoid log(0)
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)
        
        #Calculate loss 
        loss = -(y_train_tensors * torch.log(y_pred) + (1 - y_train_tensors) * torch.log(1 -  y_pred)).mean()
        return loss
    
    

###Important Parameters 

##Training Model

In [41]:
#Create Model
model = mySimplenn(X_train_tensors)

learning_rate = 0.1
epochs = 25 #How many runs of your data

#Define loop
for epoch in range(epochs):
    #Forward Pass
    y_pred = model.forward(X_train_tensors)
    
    #loss calculate 
    loss = model.loss_function(y_pred, y_train_tensors)
    
    #Backward Pass 
    loss.backward()
    
    #Update Parameters
    with torch.no_grad():
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad
    
    #Zero Gradients
    model.weights.grad.zero_()
    model.bias.grad.zero_()
    
    #Print loss in each epoch
    print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")

Epoch: 1, Loss: 3.363109098025517
Epoch: 2, Loss: 3.220485021657924
Epoch: 3, Loss: 3.0725718338937082
Epoch: 4, Loss: 2.9235230551891083
Epoch: 5, Loss: 2.76645876108244
Epoch: 6, Loss: 2.604670822336413
Epoch: 7, Loss: 2.438650352472082
Epoch: 8, Loss: 2.269300069335709
Epoch: 9, Loss: 2.1038066616920092
Epoch: 10, Loss: 1.939682251478847
Epoch: 11, Loss: 1.7769493013962514
Epoch: 12, Loss: 1.6169074436155408
Epoch: 13, Loss: 1.4623179321958093
Epoch: 14, Loss: 1.3201046284154636
Epoch: 15, Loss: 1.1943526467559422
Epoch: 16, Loss: 1.0869004853098283
Epoch: 17, Loss: 0.9987718332733613
Epoch: 18, Loss: 0.9296170677640939
Epoch: 19, Loss: 0.8774537827560247
Epoch: 20, Loss: 0.8390913992025076
Epoch: 21, Loss: 0.8110363430011733
Epoch: 22, Loss: 0.7902458176332764
Epoch: 23, Loss: 0.7744264174014317
Epoch: 24, Loss: 0.7619969852910937
Epoch: 25, Loss: 0.7519207346834049


##Evaluation

In [ ]:
#Model Evaluation
with torch.no_grad():
    y_pred = model.forward(X_test_tensors)
    y_pred = (y_pred > 0.5).float()
    accuracy = (y_pred == y_test_tensors).float().mean()
    print(f"accuracy:  {accuracy.item()}")

print(y_train_tensors)

tensor([0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1,
        0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0,
        0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0,
        1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0,
        0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0,
        0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
        1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1,
        0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1,
        1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1,
        1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0,
        0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1,